In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchsort  # pip install torchsort
from torchvision import transforms

def tensor_normaliser(_tensor):
    mean = _y_hat.mean()
    std = _y_hat.std()
    normalized_y_hat = (_y_hat - mean) / std
    return normalized_y_hat

def spearman_corr(pred, target, regularization_strength=1.0):
    # Compute differentiable soft ranks
    # pred = tensor_normaliser(pred.detach().squeeze(0)).float().unsqueeze(0)
    
    pred_ranks = torchsort.soft_rank(pred, regularization_strength=regularization_strength)
    target_ranks = torchsort.soft_rank(target, regularization_strength=regularization_strength)
    # Normalize to zero mean and unit norm
    pred_norm = (pred_ranks - pred_ranks.mean()) / pred_ranks.norm()
    target_norm = (target_ranks - target_ranks.mean()) / target_ranks.norm()
    # print(pred_norm, target_norm) # problem in pred_norm
    # print(pred, pred_ranks)
    # print(target, target_ranks)
    return (pred_norm * target_norm).sum()  # Cosine similarity ≈ correlation

In [19]:
import pandas as pd
import json
import numpy as np

_k = 3
_ret = 'e5'
_task = 'nq'
target_metric = 'f1'

if(_task=='nq'):
    _dataset_dev, _dataset_test, _prefix, _suffix = 'nq_dev', ['nq_test'], 'short', 'concise'
elif(_task=='dl'):
    _dataset_dev, _dataset_test, _prefix, _suffix = 'dev_small', ['19', '20'], 'random', 'prompt1'

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_1calls_0_0_bm25_dl_{_dataset_dev}_{_suffix}_eval.json')
zero_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}.json')
k_gens = json.load(f)
f.close()

f = open(f'../coherence_eval/log_prob_temp_res/full_context/{_dataset_dev}_{_ret}_{_k}.json')
perpC = json.load(f)
f.close()

qpp_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_combined_qpp_{_dataset_dev}.csv')

dev_res = qpp_df[['qid', 'query']].drop_duplicates().copy()

# Expand the dataframe for the convenience of analysis
for qpp_name in qpp_df.qpp_method.unique():
    value_dict = dict(zip(qpp_df[qpp_df.qpp_method==qpp_name]['qid'], qpp_df[qpp_df.qpp_method==qpp_name]['qpp_estimate']))
    dev_res[qpp_name] = dev_res.qid.apply(lambda _qid: value_dict[_qid])

if(_task=='nq'):
    base_f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items()}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items()}
    # em_dict = {item[0]: item[1]['0']['0']['EM'] for item in k_evals.items()}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items()}
elif(_task=='dl'):
    base_f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in k_evals.items() if ('0' in item[1].keys())}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

dev_res = dev_res[dev_res.qid.astype('str').isin(f1_dict.keys())]
dev_res['f1'] = dev_res.qid.apply(lambda x: f1_dict[str(x)])
dev_res['utility'] = dev_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])
# dev_res['em'] = dev_res.qid.apply(lambda x: em_dict[str(x)])
dev_res['prob(k)'] = dev_res.qid.apply(lambda x: kshot_prob_dict[str(x)])
dev_res['perpC'] = dev_res.qid.apply(lambda x: perpC[str(x)])

dev_res = dev_res.dropna(axis='index')
dev_res.head(3)

,qid,query,nqc,maxScore,spatial,a_ratio,bertQPP,bertQPP(QV),f1,utility,prob(k),perpC
0,dev_0,who sings does he love me with reba,0.003414,0.892299,2964.738525,1.076671,0.474478,0.392192,1.000000,1.000000,-0.398534,-1.401367
1,dev_1,how many pages is invisible man by ralph ellison,0.000083,0.886156,3068.109619,1.088676,0.371123,0.424146,0.000000,0.000000,-1.182933,-1.151367
2,dev_10,who is the bad guy in lord of the rings,0.000654,0.879032,2867.745850,1.038294,0.051733,0.095118,0.666667,-0.333333,-0.677929,-1.977539


In [20]:
retr_cen = dev_res[['nqc', 'spatial', 'maxScore', 'a_ratio', 'bertQPP']].apply(lambda x: np.log(x+1)).values
reader_cen = dev_res[['perpC']].values
dev_data = np.hstack((retr_cen, reader_cen))
# dev_data = dev_data.astype(np.float64)

In [21]:
dev_data.shape

(8757, 6)

In [22]:
X = torch.from_numpy(dev_data).float()

In [23]:
y = torch.from_numpy(dev_res['f1'].values).float().unsqueeze(0)

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
import torch.nn.functional as F

init_params = torch.tensor([5.97070909, 0.17282513, 1.50939513, 0.08064101, 0.12596787, 0.01269737], dtype=torch.float32)

# init_params = torch.tensor([
#     0,  1,  0,
#     0,  0, 0
# ], dtype=torch.float32)

n_features = len(init_params)
model = nn.Linear(n_features, 1, bias=False)

# Set initial weights
with torch.no_grad():
    model.weight.data = init_params.unsqueeze(0)

optimizer = optim.Adam(model.parameters(), lr=0.2)

# Your training loop
for epoch in range(1000):
    optimizer.zero_grad()
    y_hat = model(X) # assuming X shape is (1, n_features) per sample
    y_hat = y_hat.reshape(1, -1)
    y_hat = F.normalize(y_hat, p=2, dim=1)
    # print(y_hat)
    corr = spearman_corr(y_hat, y, regularization_strength=2)
    loss = - corr  # we maximize correlation
    gr = loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}: Spearman ≈ {corr.item():.4f}")
        print(model.weight.data)
        # print(y_hat.detach().cpu().numpy()[0], np.array(y[0]))
        print(stats.spearmanr(y_hat.detach().cpu().numpy()[0], np.array(y[0])))

print("Learned weights:", model.weight.data.squeeze())

Epoch 0: Spearman ≈ 0.0000
tensor([[5.9707, 0.1728, 1.5094, 0.0806, 0.1260, 0.0127]])
SignificanceResult(statistic=0.2653559664823712, pvalue=4.9472298712034393e-141)
Epoch 50: Spearman ≈ 0.0000
tensor([[5.9707, 0.1721, 1.5094, 0.0806, 0.1266, 0.0144]])
SignificanceResult(statistic=0.2656007092860714, pvalue=2.680305567188695e-141)
Epoch 100: Spearman ≈ 0.0000
tensor([[5.9707, 0.1714, 1.5094, 0.0806, 0.1272, 0.0161]])
SignificanceResult(statistic=0.2657651437898324, pvalue=1.7749660890942103e-141)
Epoch 150: Spearman ≈ 0.0000
tensor([[5.9707, 0.1706, 1.5094, 0.0806, 0.1279, 0.0179]])
SignificanceResult(statistic=0.2658159499016887, pvalue=1.5626424562380674e-141)
Epoch 200: Spearman ≈ 0.0000
tensor([[5.9707, 0.1699, 1.5094, 0.0806, 0.1285, 0.0196]])
SignificanceResult(statistic=0.26583344726842023, pvalue=1.495551195247411e-141)
Epoch 250: Spearman ≈ 0.0000
tensor([[5.9707, 0.1691, 1.5094, 0.0806, 0.1291, 0.0213]])
SignificanceResult(statistic=0.26566177880950204, pvalue=2.299970567964

In [25]:
loss

tensor(-1.0195e-13, grad_fn=<NegBackward0>)

In [9]:
y_hat

tensor([[1.0088, 2.3130, 1.7087,  ..., 1.3960, 1.1219, 0.8520]],
       grad_fn=<ViewBackward0>)

In [ ]:
F.normalize(y_hat, p=2, dim=1)

In [155]:
_y_hat = y_hat.detach().squeeze(0)
mean = _y_hat.mean()
std = _y_hat.std()
normalized_y_hat = (_y_hat - mean) / std

In [156]:
normalized_y_hat

tensor([-0.7923,  1.1300,  0.2393,  ..., -0.2217, -0.6256, -1.0236])

In [89]:
from scipy import stats
import numpy as np

y_hat = model(X).squeeze(0)
# corr = spearman_corr(y_hat, y, regularization_strength=0.5)

stats.spearmanr(y_hat.detach().numpy(), np.array(y[0]))

SignificanceResult(statistic=0.1819511416091911, pvalue=5.067864723667618e-53)

In [55]:
y

tensor([[0.5373, 0.5888, 0.0000,  ..., 0.5259, 0.4848, 0.5589]])

In [56]:
y_hat

tensor([[-4.0470],
        [-4.0568],
        [-4.2485],
        ...,
        [-3.7979],
        [-4.3885],
        [-4.3288]], grad_fn=<SqueezeBackward1>)